In [1]:
import xarray as xr
import earthaccess
import boto3
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import warnings
from IPython.display import display, Markdown
import pandas as pd
import geopandas as gpd
import rasterio
import datetime
import pyarrow as pa
import pyarrow.parquet as pq
import os

warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
## This code block is ~2 (1.5 ish) GB of memory on its own. 

fires = pd.read_parquet("s3://maap-ops-workspace/shared/zbecker/TESS_fire_spread/sigdeltas_Tess.parq")
subset_fires = gpd.read_parquet("s3://maap-ops-workspace/shared/zbecker/YANG/large_feds_faf_double_matched.parq")
subset_fires = subset_fires.to_crs(4326)

subset_fires["centroid"] = subset_fires.to_crs(4326).centroid
fires["UfireID"] = fires.mergeid.astype("int").astype("str") + "_" + fires.year.astype("str")
subset_fires["UfireID"] = subset_fires.mergeid.astype("str") + "_" + subset_fires.year.astype("str")
subset_fires["polygon"] = subset_fires.geometry
fires = fires[fires.UfireID.isin(subset_fires[subset_fires.intersectsMTBS == True].UfireID)]
fname = pd.read_csv("s3://maap-ops-workspace/shared/zbecker/Eli_MTBS_vs_FEDS/v6_output.csv")

fires = fires.merge(subset_fires[['UfireID', 'centroid', 'polygon']], on = 'UfireID' )
fires = gpd.GeoDataFrame(fires, geometry = 'polygon')

def get_st_sp_fire(df, days_after = 7):
    df.loc[:, "start_time"] = df.t.min()
    df.loc[:, "end_time"] = df.t.max()
    df.loc[:, "end_time_plus"] = df.t.astype("datetime64[ns]").max()  + datetime.timedelta(days = days_after)
    df = df.loc[df.t == df.t.max(), :]
    return(df)
    
fires_sm = fires.groupby("UfireID").apply(get_st_sp_fire).reset_index(drop = True)
fires_sm["stable_index"] = fires_sm.index

In [11]:
## Reading in my giant geoparquet

precip = gpd.read_parquet(os.path.abspath("IMERG/daily_climatology/daily_climatology")) # Not yet finished, but lets have some fun with it. 

In [ ]:
precip = precip.drop(columns=["t", "name"])
precip = precip.rename(columns={"time": "t"})

In [24]:
## TO DO 

def correct_growth_to_daily(df):
    # Add up 00:00:00 growth to 12:00:00 of the next increment

    # think about what that means for first increment

    return(df)

def correct_daily_time(df):
    ## Either add a noon value to the daily growth or drop the hour column from the "t" in fires

In [41]:
precipm = precip.merge(fires.loc[:, ["UfireID","t", "centroid"]],  on = ["UfireID"]) ## Shit I need daily growth increments. Need to merge on "t" as well.  

In [ ]:
## A quick an dirty exploratory plot

